# Genie Cache Gateway — Demo

The Databricks Genie API has a hard limit of **5 queries per minute per workspace**.  
This notebook fires 7 queries **in parallel** to show the problem and how the Gateway solves it.

**The only change between "Direct" and "Via Gateway" is the host and the ID.**

| Scenario | Host | ID | Result |
|----------|------|----|--------|
| Direct to Genie | `WORKSPACE_HOST` | `GENIE_SPACE_ID` | 429 errors (rate limited) |
| Via Gateway (1st) | `APP_HOST` | `GATEWAY_ID` | All succeed (queue + retry) |
| Via Gateway (2nd) | `APP_HOST` | `GATEWAY_ID` | Instant (semantic cache) |

In [ ]:
# Config
APP_HOST       = "https://genie-cache-queue-7474650836156271.aws.databricksapps.com"
GATEWAY_ID     = "a502f1f1-ef8c-414c-b062-cf9102643341"
GENIE_SPACE_ID = "01f11f1ae00114379e7671c8a4b8459f"

# Questions
questions = [
    "What are the top 3 nations by total revenue?",
    "How many orders were placed in January 1994?",
    "What is the average account balance of customers in the BUILDING segment?",
    "Which supplier has the most parts?",
    "Total number of lineitems with quantity greater than 40",
    "Revenue by year for the ASIA region",
    "How many distinct part types exist?",
]

In [ ]:
import requests, time, json
from concurrent.futures import ThreadPoolExecutor, as_completed
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
WORKSPACE_HOST = w.config.host
USER_EMAIL = w.current_user.me().user_name

# Token from secrets (refresh via: databricks auth token + secrets put-secret)
TOKEN = dbutils.secrets.get(scope="genie-cache", key="pat")

H = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

# --- Validate ----
assert GENIE_SPACE_ID, "Set GENIE_SPACE_ID above"
assert GATEWAY_ID, "Set GATEWAY_ID above (from app UI → Gateways)"

r = requests.get(f"{APP_HOST}/api/health", headers=H, timeout=10)
assert r.status_code == 200, f"App unreachable: {r.status_code} {r.text[:200]}"

gw = requests.get(f"{APP_HOST}/api/gateways/{GATEWAY_ID}", headers=H, timeout=10)
assert gw.status_code == 200, f"Gateway not found: {gw.status_code} {gw.text[:200]}"
gw_data = gw.json()

print(f"OK | User: {USER_EMAIL} | Space: {GENIE_SPACE_ID[:12]}... | Gateway: {gw_data['name']}")

In [ ]:
# =============================================================================
# ONE function — only host and space_id change between scenarios
# =============================================================================

def ask_genie(host, space_id, question):
    """Send a question to the Genie API. Same function for direct and gateway."""
    r = requests.post(
        f"{host}/api/2.0/genie/spaces/{space_id}/start-conversation",
        headers=H, json={"content": question}, timeout=180,
    )
    if r.status_code == 429:
        return {"status": "429", "sql": None, "from_cache": False}
    if r.status_code != 200:
        return {"status": f"HTTP {r.status_code} | {r.text[:200]}", "sql": None, "from_cache": False}

    data = r.json()
    conv_id = data.get("conversation_id", "")
    msg_id = data.get("message_id", "")

    # Poll until terminal status
    if data.get("status") not in ("COMPLETED", "FAILED", "CANCELLED"):
        for _ in range(90):
            time.sleep(2)
            r2 = requests.get(
                f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{conv_id}/messages/{msg_id}",
                headers=H, timeout=30,
            )
            if r2.status_code != 200: continue
            data = r2.json()
            if data.get("status") in ("COMPLETED", "FAILED", "CANCELLED"): break

    # Extract SQL from attachments
    sql = None
    for att in data.get("attachments", []):
        q = att.get("query", {}) if isinstance(att, dict) else {}
        sql = q.get("query") or q.get("sql")
        if sql: break

    from_cache = any("cache" in str(att.get("text", {}).get("content", "")).lower()
                      for att in data.get("attachments", []) if isinstance(att, dict))

    return {"status": data.get("status", "UNKNOWN"), "sql": sql, "from_cache": from_cache}


def run_parallel(label, host, space_id):
    """Fire all questions in parallel against a given host + space_id."""
    print(f"\n{'='*80}")
    print(f"  {label}")
    print(f"  host={host}")
    print(f"  id={space_id}")
    print(f"{'='*80}")

    results = [None] * len(questions)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=len(questions)) as pool:
        futs = {pool.submit(ask_genie, host, space_id, q): i for i, q in enumerate(questions)}
        for f in as_completed(futs):
            results[futs[f]] = f.result()
    total = time.time() - t0

    for i, q in enumerate(questions):
        r = results[i]
        tag = "CACHE" if r.get("from_cache") else ("429" if r["status"] == "429" else "GENIE")
        print(f"\n  [{i+1}] {tag:>5s} | {r['status']}")
        print(f"  Q: {q}")
        if r.get("sql"):
            print(f"  SQL: {r['sql'][:100]}")

    print(f"\n  Total: {total:.1f}s")
    return results, total

print("Functions ready.")

## Scenario 1: Direct to Genie (7 in parallel)

Same Genie Space, 7 simultaneous queries → **429 Too Many Requests** for some.

```
ask_genie(host=WORKSPACE_HOST, space_id=GENIE_SPACE_ID, question=...)
```

In [ ]:
# Scenario 1: Direct → WORKSPACE_HOST + GENIE_SPACE_ID
direct, d_time = run_parallel("Direct to Genie (7 in parallel)", WORKSPACE_HOST, GENIE_SPACE_ID)

ok = sum(1 for r in direct if r.get("sql"))
blocked = sum(1 for r in direct if r["status"] == "429")
print(f"\nResult: {ok} completed, {blocked} blocked (429)")

In [ ]:
# Scenario 2a: Gateway → APP_HOST + GATEWAY_ID  (same function, only host + id change)
app1, a1_time = run_parallel("Via Gateway, first round (7 in parallel)", APP_HOST, GATEWAY_ID)

genie_ok = sum(1 for r in app1 if not r.get("from_cache") and r.get("sql"))
cache_ok = sum(1 for r in app1 if r.get("from_cache"))
failed   = sum(1 for r in app1 if "COMPLETED" not in r["status"])
print(f"\nResult: {genie_ok} via Genie, {cache_ok} from cache, {failed} failures")

## Scenario 2b: Via Gateway — second round (all from cache)

Same queries again, same host + id. All should hit the **semantic cache** — instant response.

```
ask_genie(host=APP_HOST, space_id=GATEWAY_ID, question=...)  # same call, cached
```

In [ ]:
# Scenario 2b: Gateway again → APP_HOST + GATEWAY_ID  (expect all from cache)
app2, a2_time = run_parallel("Via Gateway, second round (cache)", APP_HOST, GATEWAY_ID)

cache_count = sum(1 for r in app2 if r.get("from_cache"))
print(f"\nResult: {cache_count}/{len(questions)} from cache | {a2_time:.1f}s vs {d_time:.1f}s direct")

## Comparison

In [ ]:
def _tag(r):
    if r.get("from_cache"): return "CACHE"
    if r["status"] == "429": return "429"
    if r.get("sql"): return "OK"
    return r["status"][:8]

print(f"\n{'='*70}")
print(f"  COMPARISON — Gateway: {gw_data['name']}")
print(f"{'='*70}")
print(f"\n  {'Question':32s} | {'Direct':>8s} | {'GW 1st':>8s} | {'GW 2nd':>8s}")
print(f"  {'-'*64}")
for i in range(len(questions)):
    print(f"  {questions[i][:30]:30s} | {_tag(direct[i]):>8s} | {_tag(app1[i]):>8s} | {_tag(app2[i]):>8s}")
print(f"  {'TIME':30s} | {d_time:>7.1f}s | {a1_time:>7.1f}s | {a2_time:>7.1f}s")

print(f"\n{'='*70}")
print(f"  SUMMARY")
print(f"{'='*70}")
print(f"  Direct:     {sum(1 for r in direct if r['status']=='429')}/{len(questions)} blocked (429)")
print(f"  GW 1st run: {sum(1 for r in app1 if r.get('sql'))}/{len(questions)} completed, zero 429s")
print(f"  GW 2nd run: {sum(1 for r in app2 if r.get('from_cache'))}/{len(questions)} from cache ({a2_time:.1f}s)")